# Линейная классификация текстов в sklearn: Logistic Regression (20 Newsgroups)



In [1]:
# --- Импорты ---
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, log_loss, ConfusionMatrixDisplay

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

SEED = 5



## Данные: 20 Newsgroups (2 класса)

Выбираем две темы (чтобы задача была бинарной):
- `sci.space`
- `rec.sport.baseball`

Далее делаем split: train / val / test.


In [2]:
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split

ds = load_dataset("data-silence/rus_news_classifier")

X_train_text = np.array(ds["train"]["news"], dtype=object)
y_train = np.array(ds["train"]["labels"])

print(len(X_train_text), len(y_train))

X_test_text  = np.array(ds["test"]["news"], dtype=object)
y_test  = np.array(ds["test"]["labels"])

X_train, X_val, y_train, y_val = train_test_split(
    X_train_text, y_train,
    test_size=0.2, random_state=SEED, stratify=y_train
)



57530 57530


In [3]:
from sklearn.model_selection import train_test_split

ds = load_dataset("data-silence/rus_news_classifier")

X_all = np.array(ds["train"]["news"], dtype=object)
y_all = np.array(ds["train"]["labels"])

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all,
    test_size=0.2, random_state=7, stratify=y_all
)

import re

def ru_preprocess(s: str) -> str:
    s = s.lower().replace("ё", "е")
    s = re.sub(r"http\S+|www\.\S+", " ", s) 
    s = re.sub(r"[^а-я\s-]+", " ", s) 
    s = re.sub(r"\s+", " ", s).strip()
    return s



## 2) Пайплайн, гиперпараметры, кросс-валидация, выбор лучшей модели


In [4]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score



pipe_lr = Pipeline([
    ("vec", TfidfVectorizer(
        preprocessor=ru_preprocess,
        token_pattern=r"(?u)\b[а-я]{2,}\b",
        ngram_range=(1,1),
        min_df=3
    )),
    ("clf", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        learning_rate="constant",
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=SEED
    ))

])

param_grid_lr = {
    "clf__alpha": [1e-5, 1e-4, 1e-3],
    "clf__eta0":  [0.01, 0.02],
    "clf__tol":   [1e-4],
    "clf__max_iter": [500],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

gs_lr = GridSearchCV(
    pipe_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv = cv,
    n_jobs=-1,
    # pre_dispatch=-1,
    refit=True,
    verbose=2
)

gs_lr.fit(X_train, y_train)

best_lr = gs_lr.best_estimator_
val_pred = best_lr.predict(X_val)
test_pred = best_lr.predict(X_test_text)

lr_val_acc = accuracy_score(y_val, val_pred)
lr_val_f1 = f1_score(y_val, val_pred, average="macro")

lr_test_acc = accuracy_score(y_test, test_pred)
lr_test_f1 = f1_score(y_test, test_pred, average="macro")

clf = best_lr.named_steps["clf"]
print("n_iter_:", clf.n_iter_, "max_iter:", clf.max_iter)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
n_iter_: 148 max_iter: 500


## 3) То же, но со второй моделью

In [5]:

from sklearn.svm import LinearSVC

pipe_svm = Pipeline([
    ("vec", TfidfVectorizer(
        preprocessor=ru_preprocess,
        token_pattern=r"(?u)\b[а-я]{2,}\b",
        ngram_range=(1,1),
        min_df=3,
    )),
    ("clf", LinearSVC())
])

param_grid_svm = {
    "clf__C": [0.1, 0.3, 1, 3, 10],
}

gs_svm = GridSearchCV(
    pipe_svm,
    param_grid=param_grid_svm,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True
)

gs_svm.fit(X_train, y_train)

best_svm = gs_svm.best_estimator_
svm_test_pred = best_svm.predict(X_test_text)

svm_test_acc = accuracy_score(y_test, svm_test_pred)
svm_test_f1  = f1_score(y_test, svm_test_pred, average="macro")

## 4) Регуляризация и вычисление самых характерных слов.

In [6]:

pipe_l1 = Pipeline([
    ("vec", TfidfVectorizer(
        preprocessor=ru_preprocess,
        token_pattern=r"(?u)\b[а-я]{2,}\b",
        ngram_range=(1,2),
        min_df=5,
        max_df=0.95,
        max_features = 200_000,
        dtype = np.float32
    )),
    ("clf", SGDClassifier(
        loss="log_loss",
        penalty="l1",
        learning_rate="constant",
        eta0=gs_lr.best_params_["clf__eta0"], 
        tol=gs_lr.best_params_["clf__tol"],
        max_iter=gs_lr.best_params_["clf__max_iter"],
        random_state=7
    ))
])

param_grid_l1 = {"clf__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]}
gs_l1 = GridSearchCV(pipe_l1, param_grid_l1, scoring="f1_macro", cv=cv, n_jobs=-1, refit=True, error_score="raise")

from joblib import parallel_backend

with parallel_backend("threading", n_jobs=4):
    gs_l1.fit(X_train, y_train)

best_l1 = gs_l1.best_estimator_

## Вывести топ-признаки

In [ ]:

import numpy as np

vec = best_l1.named_steps["vec"]
clf = best_l1.named_steps["clf"]
feat = vec.get_feature_names_out()
W = clf.coef_ 

classes_to_show = [0, 1, 2]
topk = 20

for c in classes_to_show:
    idx = np.argsort(W[c])[-topk:][::-1]
    print(f"\nCLASS {c} top features:")
    for j in idx:
        print(f"{feat[j]:<25}  {W[c, j]:.4f}")


CLASS 0 top features:
климата                    15.0709
выбросов                   9.4851
экологии                   9.2870
зоопарке                   8.6703
природы                    8.6444
заповедника                7.6480
отходов                    7.2907
спасли                     7.1179
зоопарка                   6.8649
выбросы                    6.8502
заповеднике                6.7732
рэо                        6.6781
планеты                    6.6549
потепления                 6.6490
гриб                       6.4766
явление                    6.3688
окружающей                 6.3524
изменения климата          6.2937
глобального потепления     6.2299
заметили                   6.0901

CLASS 1 top features:
фсб                        10.5870
мвд                        9.7346
ленте ру                   9.7271
ленте                      9.6199
преступлений               7.7275
этом ленте                 7.4737
оружие                     7.2133
сизо                       7.1111


## 4 Попробовать **char n‑grams TF‑IDF** и сравнить с word TF‑IDF.

In [ ]:
char_vec = TfidfVectorizer(
    preprocessor=ru_preprocess,
    analyzer="char",
    ngram_range=(3,5),
    min_df=3,
)

pipe_lr_char = Pipeline([
    ("vec", char_vec),
    ("clf", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        learning_rate="constant",
        random_state=7
    ))
])

gs_lr_char = GridSearchCV(
    pipe_lr_char, 
    param_grid_lr, 
    scoring="f1_macro", 
    cv=cv, 
    n_jobs=-1, 
    refit=True)

with parallel_backend("threading", n_jobs=8):
    gs_lr_char.fit(X_train, y_train)

best_lr_char = gs_lr_char.best_estimator_
char_test_pred = best_lr_char.predict(X_test_text)
char_test_acc = accuracy_score(y_test, char_test_pred)
char_test_f1  = f1_score(y_test, char_test_pred, average="macro")